# Gate: custom tool을 사용하는 human-in-the-loop

많은 workflow는 "완전 자동화"와 "항상 사람에게 묻기" 사이에 있습니다. 비용 승인이 전형적인 예입니다. agent는 명확한 사례를 스스로 처리할 수 있지만, 모호한 사례를 언제 사람의 검토로 넘겨야 하는지 알아야 합니다. 여기서는 판단 조정이 중요합니다. 모든 것을 escalation하는 agent와 일하는 것은 지치고, 아무것도 escalation하지 않는 agent는 위험합니다.

이 notebook은 두 개의 **custom tool**을 중심으로 비용 승인자를 만듭니다. 명확한 사례에는 `decide()`, 모호한 사례에는 `escalate()`를 사용합니다. 둘 다 애플리케이션을 왕복하며, 애플리케이션에서 결과를 기록하거나(decide), 검토자에게 제시합니다(escalate).

## custom tool이란

지금까지 cookbook은 sandbox container 내부에서 실행되는 built-in `agent_toolset`(bash, read, write 등)을 사용했습니다. **Custom tool**은 다릅니다. agent가 호출하면 세션이 일시 정지하고 `agent.custom_tool_use` event를 내보냅니다. 애플리케이션이 호출을 확인하여 원하는 코드를 실행한 뒤 `user.custom_tool_result` event를 POST로 돌려보냅니다. 세션은 이 결과를 agent의 context에 넣고 재개합니다.

이는 다음 두 상황에 적합한 형태입니다.

1. **데이터가 sandbox에서 접근할 수 없는 곳에 있습니다.** 자체 네트워크 경계 뒤에 있는 모든 것이 해당합니다. agent는 왕복 과정을 통해 애플리케이션으로 다시 호출합니다.
2. **사람이 참여하거나, 모든 호출 앞에 자체 audit 및 승인 계층을 두고 싶습니다.** 이 notebook이 하는 일입니다. `decide`와 `escalate`는 추상적인 의미의 "tool"일 뿐 아니라, 업무 로직과 사람 검토자가 agent의 일을 이어받는 접점입니다.

(다른 확장 패턴인 MCP toolset과 `resources=` repo mount는 각각 operate notebook과 orchestrate notebook에서 다룹니다.)

notebook은 두 부분으로 나뉩니다. Part A는 로컬에서 event를 streaming하고 custom tool 호출이 도착할 때마다 응답하여 세션을 진행합니다. 모든 것이 하나의 process에서 일어나고 동작을 실시간으로 볼 수 있으므로 개발 중에 편리합니다. Part B는 production webhook 패턴에 대한 짧은 안내이며, 이 패턴은 operate notebook에서 처음부터 끝까지 설명합니다.

fixture는 `example_data/gate/`에 있으며, 정책의 모든 분기를 실행해 보는 `policy.yaml`과 영수증 열두 개가 들어 있습니다.


In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

from anthropic import Anthropic
from utilities import wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()
FIXTURE = Path("example_data") / "gate"

## 1. 정책과 영수증 업로드하기


In [ ]:
policy = client.beta.files.upload(
    file=("policy.yaml", (FIXTURE / "policy.yaml").read_bytes(), "text/yaml")
)
receipts = client.beta.files.upload(
    file=(
        "receipts.jsonl",
        (FIXTURE / "inbox" / "receipts.jsonl").read_bytes(),
        "application/jsonl",
    )
)

## 2. custom tool 두 개를 가진 agent 정의하기

custom tool은 built-in toolset과 같은 `tools=` 배열에 `"type": "custom"` 및 입력용 JSON schema로 선언합니다. 각 선언은 model에 tool의 용도(`description`), 호출 시 전달할 내용(`input_schema`), 이름을 알려 줍니다. agent는 언제 호출할지 결정하고, 코드는 호출되었을 때 무엇을 할지 결정합니다.

여기서는 built-in `agent_toolset_20260401`도 활성화하여 agent가 정책 파일과 영수증을 직접 읽을 수 있도록 합니다. `decide`와 `escalate`는 모든 결정을 왕복 과정으로 만드는 두 custom tool입니다.


In [ ]:
agent = client.beta.agents.create(
    name="cookbook-gate",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-gate-human-in-the-loop"},
    model=MODEL,
    system=(
        "You are an expense approver. Read each receipt in "
        "receipts.jsonl against the policy in policy.yaml and make "
        "exactly ONE tool call per receipt. Call decide(receipt_id, "
        "action, reason) for clear cases, or escalate(receipt_id, "
        "question) for ambiguous ones (near thresholds, unclear "
        "categories, suspicious notes). Once you've called decide "
        "or escalate for a given receipt, that receipt is finalized "
        "— do not call either tool for it again. After processing "
        "all receipts exactly once, stop."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        },
        {
            "type": "custom",
            "name": "decide",
            "description": "Record a final approve/reject for a clear-cut receipt.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "receipt_id": {"type": "string"},
                    "action": {"type": "string", "enum": ["approve", "reject"]},
                    "reason": {"type": "string"},
                },
                "required": ["receipt_id", "action", "reason"],
            },
        },
        {
            "type": "custom",
            "name": "escalate",
            "description": "Surface an ambiguous receipt for human review.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "receipt_id": {"type": "string"},
                    "question": {"type": "string"},
                },
                "required": ["receipt_id", "question"],
            },
        },
    ],
)

env = client.beta.environments.create(
    name="cookbook-gate-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[
        {"type": "file", "file_id": policy.id, "mount_path": "policy.yaml"},
        {"type": "file", "file_id": receipts.id, "mount_path": "receipts.jsonl"},
    ],
    title="Expense gate",
)
print(f"session: {session.id}")

## Part A: 개발 중 로컬 streaming

custom tool agent를 진행시키는 가장 간단한 방법은 세션의 event를 streaming하고 tool 호출이 도착할 때마다 반응하는 것입니다. `decide` 호출은 기록하고 `escalate` 호출에는 그 자리에서 모의 인간 결정을 제공합니다. production에서는 escalation을 queue에 넣고 실제 검토자가 나중에 처리하게 하며, operate notebook에서 이를 다룹니다.


In [ ]:
def simulate_human_review(receipt_id: str, question: str) -> str:
    # 실제 구현은 UI에 표시하고 입력을 기다립니다.
    # 여기서는 agent가 의심스럽다고 표시한 모든 것을 거부합니다.
    return "reject" if "suspicious" in question.lower() else "approve"


# iterate notebook은 streaming 루프를
# `stream_until_end_turn`으로 분리했고 대부분의 다른 notebook은 import만 합니다.
# 이 notebook은 그러지 않습니다. agent가 하는 모든 결정이
# custom tool 호출이므로 세션은 계속 idle이 되며
# `stop_reason.type == "requires_action"`이고
# `stop_reason.event_ids`는 응답을 기다리는 `agent.custom_tool_use`
# event를 가리킵니다. 각각 `user.custom_tool_result`를 POST하고
# 세션이 재개하게 한 뒤, 최종적으로
# `end_turn`과 함께 도착하는 `session.status_idle`에서 종료합니다. helper는
# `end_turn`에서 종료하는 것만 알고 있으므로 여기에 전체 루프가
# 필요합니다.
decisions = {}  # receipt_id -> 최종 결정 레코드
tool_use_events = {}
responded_to = set()  # 이미 응답한 event_id
print("=== Part A: streaming ===")
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Read /mnt/session/uploads/policy.yaml and "
                            "/mnt/session/uploads/receipts.jsonl. Process "
                            "all 12 receipts. For each receipt, make "
                            "exactly one decide() or escalate() call and "
                            "then move on to the next. When every receipt "
                            "has been processed once, stop."
                        ),
                    }
                ],
            }
        ],
    )
    # responded_to 집합 참고: agent가 병렬 custom tool 호출을 5개 넘게 내보내면
    # 서버는 `stop_reason.event_ids`를 다음 대기 중 5개를 담은
    # sliding window로 반환합니다.
    # stream에서 관찰하는 각 status_idle에는
    # event가 발생한 순간의 window가 고정되어 있지만
    # 다음 status_idle event로 넘어갈 때는 서버가
    # 방금 응답한 event를 이미 지나갔습니다. 따라서
    # 같은 custom tool 호출에 두 번 응답하지 않도록 status_idle event 사이에서
    # 중복을 제거해야 합니다(중복 응답은 400).
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            tool_use_events[ev.id] = ev
        elif ev.type == "session.status_idle" and ev.stop_reason:
            if ev.stop_reason.type == "requires_action":
                for event_id in ev.stop_reason.event_ids:
                    if event_id in responded_to:
                        continue
                    tool_ev = tool_use_events[event_id]
                    name, args = tool_ev.name, tool_ev.input
                    receipt_id = args["receipt_id"]
                    if name == "decide":
                        decisions[receipt_id] = {"lane": args["action"], **args}
                        result = {"recorded": True}
                    elif name == "escalate":
                        human = simulate_human_review(receipt_id, args["question"])
                        decisions[receipt_id] = {
                            "lane": "escalated",
                            "human_decision": human,
                            **args,
                        }
                        result = {"human_decision": human}
                    else:
                        result = {"error": f"unknown tool {name}"}
                    client.beta.sessions.events.send(
                        session_id=session.id,
                        events=[
                            {
                                "type": "user.custom_tool_result",
                                "custom_tool_use_id": event_id,
                                "content": [{"type": "text", "text": json.dumps(result)}],
                            }
                        ],
                    )
                    responded_to.add(event_id)
            elif ev.stop_reason.type == "end_turn":
                break
        elif ev.type == "session.status_terminated":
            break

wait_for_idle_status(client, session.id)

lanes = Counter(d["lane"] for d in decisions.values())
print(f"\n{len(decisions)} decisions: {dict(lanes)}")

client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")

## Part B: production용 webhook

로컬 streaming 패턴은 개발 중에는 잘 작동하지만 사람이 생각하는 동안 HTTP 연결을 열어 두므로 규모를 늘리기 어렵습니다. 대신 production 패턴은 Console에 `session.status_idled`에서 발생하는 webhook을 등록합니다. 이는 agent가 완료되었거나 tool 결과를 기다리고 있다는 신호입니다. 서버는 event를 검사하고, 대기 중인 escalation을 검토자에게 제시한 다음, 사람이 완료할 때마다 `user.custom_tool_result`를 POST로 돌려보냅니다. 서버 쪽에는 장시간 유지하는 연결이 없습니다.

operate notebook에서는 전체 webhook 설정을 처음부터 끝까지 설명합니다. Console 등록, HMAC 서명 검증, FastAPI handler, `events.send`로 돌아오는 왕복 과정이 포함됩니다. agent에 응답하는 코드는 위의 Part A와 동일하며, trigger만 바뀝니다(streaming pull 대신 webhook push).
